In [1]:
# Skirtas ecg vaizdavimui ir triukšmų parametrų skaičiavimui .

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0'

# Dar reikia pridėti informaciją iš json failo

import numpy as np
import pandas as pd
import time, sys, os, json, math
from pathlib import Path
from scipy.stats import zscore
from scipy.signal import butter, filtfilt

# Prikabiname zive_util_ml, use_ecg_denoising_util, kurie yra lygiagrečiame aplanke SUPL_FUNCTIONS
# === Išoriniai moduliai (lygiagretus aplankas) =================================
PARALLEL_PATH = '~/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/SUPL_FUNCTIONS'
sys.path.append(str(PARALLEL_PATH))

# naudosime funkcijas iš zive_data_read_utils.py
from zive_data_read_utils import load_ecg_npy, list_ecg_records, read_json_file

from ecg_denoising_util import get_ecg_signal, convert_seconds_to_hms, divide_signal_into_fragments
from ecg_denoising_util import plot_gap_legend, plot_signal_write_plot_R_P_1, get_ecg_noise_indices_annotated
from ecg_denoising_util import read_df_annot, extract_metadata_from_json, plot_signal
from use_ecg_denoising_util import ecg_filter

from classify_ecg_noise import classify_ecg_noise_segment

def get_recording_id(df, filename):
    base_name = Path(filename).stem  # Extract '1001_0' from '1001_0.npy'
    match = df[df['filename'] == base_name]
    
    if not match.empty:
        return match.iloc[0]['recordingId']
    else:
        return None  # Or raise an exception if preferred
    
def get_basename(df, filename):
    base_name = Path(filename).stem  # Extract '1001_0' from '1001_0.npy'
    match = df[df['filename'] == base_name]
    
    if not match.empty:
        basename = match.iloc[0]['basename']
        return basename.replace(',', '.')
    else:
        return None  # Or raise an exception if preferred
    
    
import warnings

# Option 1: Suppress all UserWarnings
warnings.filterwarnings("ignore", category=UserWarning)

# Option 2: Suppress only the specific warning from scikit-learn
# If you want to suppress only that specific warning related to feature names (safer than suppressing all UserWarnings):
# warnings.filterwarnings("ignore", message="X has feature names, but StandardScaler was fitted without feature names")

# print options
np.set_printoptions(threshold=sys.maxsize)

# Set display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)        # Set width to a high value to avoid line breaks
pd.set_option('display.max_colwidth', None) # Don't truncate column content

    
            # ++++++++++++++++++++++++++++++++++++++++++++  ĮVAIRŪS PARAMETRAI SKAIČIAVIMUI

print("\n\nĮVAIRŪS PARAMETRAI SKAIČIAVIMUI")

fs = 200  # Sampling frequency in Hz
print("fs:", fs)

# ECG įrašo filtravimui
fp = {  'type': 'lowpass',
            'method':'butterworth',
            'order':5,
            'sampling_rate':fs,
            'lowcut':0.5,
            'highcut':90 }

# Initialize the results dictionary
results = {
    'status': {
    'success': True,  # Assume success initially
    'error': None     # No error initially
        }
}   

#                            ++++++++++++++++++++++++++++++++++++++++++++  ECG NUSKAITYMAS IR PRADINIS APDOROJIMAS

# start1

# Žiūrimas
fileNames = ['1028_1.npy']

# Demonstarcijai švarių, vidutinių ir nešvarių
fileNames = ['1001_2.npy']


# ++++++++++++++++++++++++++++++ Testavimui
Duomenu_aplankas = Path.home() / "DI/2025_ZIVEO/DUOMENYS_ANOTUOTI/AtsisiuntimasZiveDuomenu/Atsisiusti_visi_anotuoti_duomenys_26_04_14"

# # Aplankas su EKG įrašais  ir anotacijomis (.json)
db_folder = 'Atsisiusti_visi_anotuoti_duomenys_26_04_14'

# # Nuoroda į aplanką su EKG įrašais ir metaduomenimis (.json)
rec_dir = Path(Duomenu_aplankas, db_folder)

# fileNames = ['1001_8.npy']
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++


# Read the Excel file
# file_name = "visi_zive_irasai.xlsx" 
# file_path = os.path.join(rec_dir, file_name) 
# list_df = pd.read_excel(file_path, dtype=str)
# list_df = None


portion_length_in_secs = 10

print(f"\nDuomenu_aplankas: {Duomenu_aplankas}")
print(f"db_folder: {db_folder}")
print(f"rec_dir: {rec_dir}")
print("\nFiltravimo parametrai:", fp)

print(fileNames)


for fileName in fileNames:

    start_time_b = time.time()
    start_time = time.time()

    # Initialize variables before the try block
    ecg_signal = None
    filtered_signal = None
    recID = None
    noise_indices_annotated_secs = None
    annot_df = None


    try:

                    # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ NUSKAITOME EKG SIGNALĄ
    
        filePath = os.path.join(rec_dir, fileName)             
        ecg_signal_orig = np.array(get_ecg_signal(filePath))
         
        print(f"\nFailas: {filePath}") 
        # ecg_signal_orig = np.array([])   # testas  
        if len(ecg_signal_orig) == 0:
            raise ValueError("The ECG signal is empty.")
        
        len_ecg_signal_orig_secs = len(ecg_signal_orig)/fs
        print(f"\nlen(ecg_signal_orig): {len(ecg_signal_orig)} ({len_ecg_signal_orig_secs:.1f} secs)")
        hours, minutes, seconds = convert_seconds_to_hms(len_ecg_signal_orig_secs)
        print(f"Hours: {hours:.1f}, Minutes: {minutes:.1f}, Seconds: {seconds:.1f}")

        filtered_signal = ecg_filter(ecg_signal_orig, fp)
        # filtered_signal = ecg_signal_orig
        
        # Normalize
        # ecg_signal = (filtered_signal - np.mean(filtered_signal)) / np.std(filtered_signal)
        ecg_signal = filtered_signal
        
        json_path = os.path.splitext(filePath)[0] + '.json'
        metadata = extract_metadata_from_json(json_path)
        print(f"\nmetadata: {metadata}")
        recID = metadata.get('recordingId')
        
        noise_indices_annotated = get_ecg_noise_indices_annotated(json_path)
        # noise_indices = []
        
        # Convert indices to seconds (fs = 200 Hz)
        noise_indices_annotated_secs = [(start / fs, end / fs) for start, end in noise_indices_annotated]

        # Print results
        print("\nNoise indices annotated:", noise_indices_annotated)
        formatted = [(f"{start:.1f}", f"{end:.1f}") for start, end in noise_indices_annotated_secs]
        print("Noise indices annotated secs:", formatted)

        # recID = get_recording_id(list_df, fileName)
        # basename = get_basename(list_df, fileName)
        # print(f"basename: {basename} recID: {recID}")

        annot_df = read_df_annot(rec_dir, fileName)
        # print(f"\nlen(annot_df): {len(annot_df)}")
        # print("\nannot_df:")
        # print(annot_df.head(10))
        
        
    # Klaidos skripto blokuose
    except ValueError as error:
        results['status']['success'] = False
        results['status']['error'] = str(error)

    # Unexpected error during script execution 
    except Exception as error:
        results['status']['success'] = False
        results['status']['error'] = f"Unexpected error: {str(error)}"

    print("\n", results)


                # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ VAIZDUOJAME EKG SIGNALĄ
    

    portion_length = portion_length_in_secs * fs   

    # bendras aplankas grafikams
    plot_save_dir = "PLOTTINGS_4"

    # Extract the numerical part from the fileName
    base_name = os.path.basename(fileName)

    flag_plot_save = False

    addition_to_name = 'orig'
    if flag_plot_save:
        plot_dir = os.path.join(plot_save_dir, base_name + '_plot_' + addition_to_name)
        if not os.path.exists(plot_dir):
            os.makedirs(plot_dir)
    else:
        plot_dir = None

    print(f"\nplot_dir: {plot_dir}")
    # print(f"filtered_signal: {filtered_signal}")
    if ecg_signal is not None:
        print(f"filtered_signal length: {len(ecg_signal)} ({len(ecg_signal)/fs:.1f} secs)")
    else:
        print("ecg_signal is None, cannot print length.")
    print(f"portion_length: {portion_length} ({portion_length_in_secs} secs)")
    print(f"portion_length_in_secs: {portion_length_in_secs} ({portion_length} samples)")
    # print(f"fs: {fs} ({1/fs:.1f} secs)")

    # filtered_signal = filtered_signal[:126999]

    # Sudalijame  ecg_signal_start į fragmentus
    show_frag_indices = divide_signal_into_fragments(filtered_signal, portion_length)
    print()
    print(f"show_frag_indices: {show_frag_indices}")
    show_frag_indices_secs = [(x/fs, math.floor(y/fs)) for x, y in show_frag_indices] # indexes
    print(f"show_frag_indices_secs: {show_frag_indices_secs}")

    num_fragment = 1
    for (show_frag_start_secs, show_frag_end_secs) in show_frag_indices_secs:
        
        if not flag_plot_save:
            print(f"\nFRAGMENT NR. {num_fragment}")
            print(f"plot_signal_from_in_secs: {show_frag_start_secs}")
            print(f"plot_signal_to_in_secs: {show_frag_end_secs}")
            
        # if (show_frag_end_secs - show_frag_start_secs) > 0.5:
        #     plot_signal(fileName,  ecg_signal, fs, num_fragment, show_frag_start_secs, show_frag_end_secs,\
        #                 plot_dir, addition_to_name, recID= recID,
        #                 gap1_indices_secs=noise_indices_annotated_secs,
        #                 gap2_indices_secs=[], gap3_indices_secs=[],
        #                 annot_df=annot_df, flag_secs=True)
            
            
        if (show_frag_end_secs - show_frag_start_secs) > 0.5:
            plot_signal_write_plot_R_P_1(fileName,  ecg_signal, fs, num_fragment, show_frag_start_secs, show_frag_end_secs, portion_length_in_secs,\
                        plot_dir, addition_to_name, recID= recID,
                        gap1_indices_secs=noise_indices_annotated_secs,
                        gap2_indices_secs=[], gap3_indices_secs=[],
                        annot_df=annot_df, flag_secs=False)
                
        # plot_signal_write_plot_R_P_1(fileName, ecg_signal, fs, num_fragment, plot_signal_from_in_secs, plot_signal_to_in_secs, portion_length_in_secs, 
        #                      plot_save_dir=None, save_mark=None, recID=None,
        #                      gap1_indices_secs=[], gap2_indices_secs=[],
        #                      gap3_indices_secs=[], mark_indices_secs=[],
        #                      annot_df=None, 
        #                      rpeak_indices_secs=[], ppeak_indices_secs=[],
        #                      flag_secs=True)
                
        num_fragment += 1
    
    print(f"\nPabaiga\n")



ModuleNotFoundError: No module named 'zive_data_read_utils'